# Rohdaten herunterladen (einmalig, ausserhalb der Notebook-Reihe)

**Ziel.** Alle Rohdaten, die die Notebooks 00 bis 13 lesen, einmal in den Ordner `daten/roh/` laden und jede Datei im Manifest eintragen.

**Warum.** Die Rohdaten liegen nicht im Repository: Sie sind gross (rund 30 GB für Kreis 1) und gehören den Anbietern. Wer nachrechnen will, holt sie mit diesem Notebook von den Originalquellen. Es lädt nur, was fehlt; ein zweiter Lauf lädt nichts mehr.

**Eingaben:** `einstellungen.json` (Gebiet `KREISE` und Koordinatensystem aus Notebook 00, Abschnitt 1) und eine Internetverbindung.
**Ausgaben:** die Dateien in `daten/roh/` und die Liste `daten/roh/MANIFEST.csv`.

**Reihenfolge beim ersten Mal:** Notebook 00, Abschnitt 1 (Einstellungen) → dieses Notebook → Notebook 00 ganz.

**Was geladen wird**

| Daten | Umfang | gelesen von | Quelle |
| --- | --- | --- | --- |
| Stadtkreise | ganze Stadt | Notebook 00 | Stadt Zürich (CC0) |
| DAV-Parkfelder (Punkte) und Parkstreifen (Linien), Stand 2025 | ganze Stadt | Notebook 01, Kontrollen der Ground Truth | Stadt Zürich, Dienstabteilung Verkehr (CC0) |
| Öffentlich zugängliche Strassenparkplätze, Stand 31.12.2021 | ganze Stadt | Notebooks 09 und 13: amtliche Daten zum Bild 2021 | Stadt Zürich, Tiefbauamt (CC0) |
| Amtliche Vermessung: Bodenbedeckung und Strassenstücke | ganze Stadt | Notebook 00 | Stadt Zürich, Geomatik + Vermessung (CC0) |
| Amtliche Vermessung: Liegenschaften | Gebiet mit 250 m Rand | Notebooks 01 und 04 (Strassenraum oder übriger Grund) | Stadt Zürich, Geomatik + Vermessung (CC0) |
| Liegenschafteninventar: Grundstücke im Eigentum der Stadt | ganze Stadt | Notebook 01 | Stadt Zürich (CC0) |
| Orthofoto Sommer 2024/25, 5 cm | Kacheln über dem Gebiet | Versuch B | Kanton Zürich (Quelle nennen) |
| Orthofoto Frühjahr 2021, 5 cm, ohne Laub | Kacheln über dem Gebiet | Versuch A | Kanton Zürich (Quelle nennen) |
| Orthofoto Sommer 2026, 10 cm | Kacheln über dem Gebiet | nur Notebook 01 (Vergleich dreier Aufnahmen) | Kanton Zürich (Quelle nennen) |

Die Vektordaten der Stadt werden für die **ganze Stadt** geladen: Wer später ein anderes Gebiet wählt, muss sie nicht nachladen. Die Luftbilder sind für die ganze Stadt über 100 GB gross; darum nur die Kacheln über dem Gebiet.

**Manifest.** Jede geladene Datei bekommt eine Zeile in `daten/roh/MANIFEST.csv`: Titel, Herausgeber, Adresse, Lizenz, Datum, Grösse und ein Fingerabdruck (SHA-256). Mit dem Fingerabdruck prüft Notebook 01, dass die Datei unverändert ist.

In [1]:
import datetime                    # Datum des Herunterladens
import hashlib                     # Fingerabdruck (SHA-256) einer Datei
import json                        # Einstellungen lesen
import math                        # abrunden auf ganze Kilometer
import re                          # eine Zahl aus der Antwort des Dienstes lesen
from pathlib import Path           # Pfade zusammensetzen

import requests                    # Dateien aus dem Internet laden
import pandas as pd                # Tabellen
import geopandas as gpd            # Tabellen mit Geometrie
from shapely.geometry import box   # Rechteck (Kachel)

ROOT = Path.cwd().parent                                   # Projektordner (eine Ebene über notebooks/)
ROH = ROOT / "daten" / "roh"                               # Rohdaten (werden nie verändert)
MANIFEST = ROH / "MANIFEST.csv"                            # Liste aller geladenen Dateien
EINSTELLUNGEN = json.loads((ROOT / "einstellungen.json").read_text(encoding="utf-8"))   # aus Notebook 00
KREISE, CRS = EINSTELLUNGEN["KREISE"], EINSTELLUNGEN["CRS"]
print("Gebiet: Kreise", KREISE, "| Koordinatensystem:", CRS)

Gebiet: Kreise [1] | Koordinatensystem: EPSG:2056


## 1. Drei Werkzeuge

**Schritt 1: eine Datei laden, wenn sie fehlt.** Die Funktion `hole` prüft zuerst, ob die Datei schon da ist. Wenn nicht, lädt sie sie, rechnet den Fingerabdruck und trägt die Datei ins Manifest ein. Sie gibt `True` zurück, wenn sie geladen hat.

Steht im Manifest schon eine Zeile zu dieser Datei, wird sie ersetzt. Das ist nach dem Klonen des Repositorys der Fall: Das mitgelieferte Manifest führt die Dateien des Rechners, auf dem gerechnet wurde. Hat der Anbieter eine Datei seither nachgeführt, hat die neu geladene eine andere Grösse; mit zwei Zeilen würde Notebook 01 für die alte einen Fehler melden.

Vorher: Datei fehlt → nachher: Datei in `daten/roh/` und genau eine Zeile dazu im Manifest.

In [2]:
def hole(datei, url, titel, herausgeber, lizenz):
    # Lädt eine Datei nach daten/roh/, wenn sie noch fehlt, und trägt sie ins Manifest ein
    ziel = ROH / datei
    if ziel.exists():
        print("vorhanden:", datei)
        return False
    print("lade:", datei)
    antwort = requests.get(url, timeout=1800, headers={"User-Agent": "parkplatz-pilot"})
    antwort.raise_for_status()                                       # Fehler des Servers -> Abbruch
    if "text/html" in antwort.headers.get("content-type", ""):
        raise RuntimeError("Der Server lieferte eine Webseite statt Daten: " + url)
    ziel.parent.mkdir(parents=True, exist_ok=True)
    ziel.write_bytes(antwort.content)
    with ziel.open("rb") as f:
        fingerabdruck = hashlib.file_digest(f, "sha256").hexdigest()  # SHA-256 der gespeicherten Datei
    zeile = dict(datei=f"daten/roh/{datei}", originaltitel=titel, herausgeber=herausgeber, url=url, lizenz=lizenz,
                 stand="", heruntergeladen=datetime.date.today().isoformat(), bytes=ziel.stat().st_size, sha256=fingerabdruck)
    manifest = pd.read_csv(MANIFEST) if MANIFEST.exists() else pd.DataFrame()
    if len(manifest):
        manifest = manifest[manifest.datei != zeile["datei"]]         # Maske: eine ältere Zeile zu dieser Datei fällt weg
    pd.concat([manifest, pd.DataFrame([zeile])], ignore_index=True).to_csv(MANIFEST, index=False)
    return True

**Schritt 2: die Adresse beim Webdienst der Stadt.** Die Stadt Zürich liefert ihre Daten über einen Webdienst (WFS, Web Feature Service): Man schreibt in eine Internet-Adresse, *welche* Ebene man will, und bekommt sie als Datei zurück. Mit einem Rahmen (links, unten, rechts, oben in Metern) kommt nur der Ausschnitt.

In [3]:
def wfs_adresse(dienst, ebene, rahmen=None):
    # Adresse für eine Ebene eines WFS-Dienstes der Stadt Zürich, in LV95, als GeoJSON; mit Rahmen nur der Ausschnitt
    adresse = f"https://www.ogd.stadt-zuerich.ch/wfs/geoportal/{dienst}?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&TYPENAME={ebene}&SRSNAME=EPSG:2056"
    if rahmen is not None:
        adresse += "&BBOX=" + ",".join(str(round(wert)) for wert in rahmen) + ",EPSG:2056"   # auf ganze Meter gerundet
    return adresse + "&outputFormat=application/vnd.geo%2Bjson"

print(wfs_adresse("Stadtkreise", "adm_stadtkreise_a"))

https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Stadtkreise?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&TYPENAME=adm_stadtkreise_a&SRSNAME=EPSG:2056&outputFormat=application/vnd.geo%2Bjson


**Schritt 3: Ist die Datei vollständig?** Ein Download kann abbrechen. Die Funktion fragt den Dienst, wie viele Objekte die Ebene hat (`resultType=hits` liefert nur die Zahl), und vergleicht mit der gespeicherten Datei.

In [4]:
def pruefe_vollstaendig(datei, adresse):
    # Fragt den Dienst, wie viele Objekte es gibt, und vergleicht mit der Datei (Schutz vor abgeschnittenen Downloads)
    anzahl_datei = len(gpd.read_file(ROH / datei))
    antwort = requests.get(adresse.replace("&outputFormat=application/vnd.geo%2Bjson", "&resultType=hits"), timeout=120)
    anzahl_dienst = int(re.search(r'numberOfFeatures="(\d+)"', antwort.text).group(1))
    print(f"   {anzahl_datei} Objekte, Dienst meldet {anzahl_dienst}:", "vollständig" if anzahl_datei == anzahl_dienst else "ACHTUNG, unvollständig")

## 2. Vektordaten der Stadt

**Schritt 1: sieben Ebenen für die ganze Stadt.** Die Bodenbedeckung der ganzen Stadt ist etwa 400 MB gross; der Download dauert einige Minuten.

Vorher: leerer Ordner (oder schon vorhandene Dateien) → nachher: sieben Dateien in `grenzen/`, `parkplaetze/`, `vermessung/` und `liegenschaften/`.

In [5]:
DAV = "oeffentlich_zugaengliche_Parkplaetze_DAV"
AV = "Amtliche_Vermessungsdaten_Stadt_Zuerich_Jahresendstand_2025"
AV_TITEL, AV_HG = "Amtliche Vermessungsdaten Stadt Zürich Jahresendstand 2025", "Stadt Zürich, Geomatik + Vermessung"
DAV_TITEL, DAV_HG = "öffentlich zugängliche Parkplätze DAV", "Stadt Zürich, Sicherheitsdepartement, Dienstabteilung Verkehr"
vektoren = [                                   # Datei, Dienst, Ebene, Titel, Herausgeber
    ("grenzen/adm_stadtkreise_a.geojson", "Stadtkreise", "adm_stadtkreise_a", "Stadtkreise", "Stadt Zürich"),
    ("parkplaetze/DAV_oeff_strassenparkierung_dav_p.geojson", DAV, "oeff_strassenparkierung_dav_p", DAV_TITEL, DAV_HG),
    ("parkplaetze/DAV_oeff_strassenparkierung_dav_l.geojson", DAV, "oeff_strassenparkierung_dav_l", DAV_TITEL, DAV_HG),
    ("parkplaetze/Strassenparkplaetze_OGD_2021_view_pp_ogd.geojson", "Oeffentlich_zugaengliche_Strassenparkplaetze_OGD", "view_pp_ogd",
     "Öffentlich zugängliche Strassenparkplätze OGD", "Stadt Zürich, Tiefbauamt"),
    ("vermessung/av_bo_boflaeche_a_stadt.geojson", AV, "av_bo_boflaeche_a", AV_TITEL, AV_HG),
    ("vermessung/av_geb_strassenstueck_stadt.geojson", AV, "av_geb_strassenstueck", AV_TITEL, AV_HG),
    ("liegenschaften/v_lvz_grundstueck.geojson", "Liegenschafteninventar_OGD", "v_lvz_grundstueck", "Liegenschafteninventar OGD", "Stadt Zürich"),
]
for datei, dienst, ebene, titel, herausgeber in vektoren:
    adresse = wfs_adresse(dienst, ebene)
    if hole(datei, adresse, titel, herausgeber, "CC0 (Open Data Zürich)"):
        pruefe_vollstaendig(datei, adresse)

vorhanden: grenzen/adm_stadtkreise_a.geojson
vorhanden: parkplaetze/DAV_oeff_strassenparkierung_dav_p.geojson
vorhanden: parkplaetze/DAV_oeff_strassenparkierung_dav_l.geojson
vorhanden: parkplaetze/Strassenparkplaetze_OGD_2021_view_pp_ogd.geojson
vorhanden: vermessung/av_bo_boflaeche_a_stadt.geojson
vorhanden: vermessung/av_geb_strassenstueck_stadt.geojson
vorhanden: liegenschaften/v_lvz_grundstueck.geojson


**Schritt 2: der Rahmen des Gebiets.** Aus den Stadtkreisen wird das Gebiet gebildet und um 250 m erweitert (derselbe Rand wie in Notebook 00). Der Rahmen ist das kleinste Rechteck darum; ihn brauchen die Liegenschaften und die Luftbilder.

Vorher: zwölf Stadtkreise → nachher: ein Rahmen (links, unten, rechts, oben in Metern).

In [6]:
kreise = gpd.read_file(ROH / "grenzen/adm_stadtkreise_a.geojson").to_crs(CRS)
rand = kreise[kreise["name"].isin(KREISE)].union_all().buffer(250)          # Gebiet + 250 m
links, unten, rechts, oben = rand.bounds                                     # kleinste und grösste x und y des Randes
print(f"Vorher: {len(kreise)} Stadtkreise → nachher: Rahmen {round(links)}, {round(unten)}, {round(rechts)}, {round(oben)} "
      f"({(rechts - links) / 1000:.2f} × {(oben - unten) / 1000:.2f} km)")

Vorher: 12 Stadtkreise → nachher: Rahmen 2682123, 1246154, 2684258, 1248847 (2.14 × 2.69 km)


**Schritt 3: Liegenschaften, nur im Rahmen.** Die Liegenschaften (Grundstücke) der amtlichen Vermessung sind für die ganze Stadt sehr gross. Notebook 04 braucht sie nur im Gebiet, um Strassenraum und übrigen Grund zu trennen. Der Dateiname enthält `kreis1`, weil die Notebooks 01 und 04 ihn so lesen.

Vorher: Rahmen → nachher: eine Datei mit den Liegenschaften im Rahmen.

In [7]:
LIEGENSCHAFTEN = "vermessung/av_li_liegenschaft_a_kreis1.geojson"
adresse = wfs_adresse(AV, "av_li_liegenschaft_a", rahmen=(links, unten, rechts, oben))
if hole(LIEGENSCHAFTEN, adresse, AV_TITEL, AV_HG, "CC0 (Open Data Zürich)"):
    pruefe_vollstaendig(LIEGENSCHAFTEN, adresse)
print(len(gpd.read_file(ROH / LIEGENSCHAFTEN)), "Liegenschaften in der Datei")

vorhanden: vermessung/av_li_liegenschaft_a_kreis1.geojson


5860 Liegenschaften in der Datei


## 3. Luftbilder über dem Gebiet

**Schritt 1: Welche Kacheln braucht es?** Der Kanton liefert das Orthofoto vom Sommer in Kacheln von 1 × 1 km. Der Dateiname ist die **linke untere Ecke in Kilometern**: `2683_1247` liegt von x = 2'683'000 bis 2'684'000 m und y = 1'247'000 bis 1'248'000 m. Wir gehen alle Kilometer-Quadrate im Rahmen durch und behalten die, die das Gebiet samt Rand berühren.

Vorher: Rahmen → nachher: Liste der Kachelnamen.

In [8]:
kacheln = []
for x in range(math.floor(links / 1000), math.floor(rechts / 1000) + 1):          # Kilometer von links bis rechts
    for y in range(math.floor(unten / 1000), math.floor(oben / 1000) + 1):        # Kilometer von unten bis oben
        quadrat = box(x * 1000, y * 1000, x * 1000 + 1000, y * 1000 + 1000)       # das Quadrat dieser Kachel
        if quadrat.intersects(rand):                                              # berührt es das Gebiet samt Rand?
            kacheln.append(f"{x}_{y}")
print(len(kacheln), "Kacheln:", kacheln)

9 Kacheln: ['2682_1246', '2682_1247', '2682_1248', '2683_1246', '2683_1247', '2683_1248', '2684_1246', '2684_1247', '2684_1248']


**Schritt 2: Sommer 2024/25 (Versuch B).** Je Kachel zwei Dateien: `.tif` ist das Bild (rund 0,9 GB), `.tfw` beschreibt seine Lage.

Vorher: Liste der Kacheln → nachher: die Dateien in `orthofoto/sommer_2024_25/`.

In [9]:
KANTON = "Kanton Zürich, Amt für Raumentwicklung, Geoinformation"
LIZENZ_KANTON = "Open use, Quelle nennen (Kanton Zürich)"
ORTHO_2024 = "https://maps.zh.ch/download/orthofoto/sommer/2024/tif/"
for kachel in kacheln:
    for endung in (".tif", ".tfw"):
        hole(f"orthofoto/sommer_2024_25/{kachel}{endung}", ORTHO_2024 + kachel + endung,
             "Orthofoto Sommer RGB/Infrarot 2024/25 (OGD) (kantonaler Datensatz)", KANTON, LIZENZ_KANTON)

vorhanden: orthofoto/sommer_2024_25/2682_1246.tif
vorhanden: orthofoto/sommer_2024_25/2682_1246.tfw
vorhanden: orthofoto/sommer_2024_25/2682_1247.tif
vorhanden: orthofoto/sommer_2024_25/2682_1247.tfw
vorhanden: orthofoto/sommer_2024_25/2682_1248.tif
vorhanden: orthofoto/sommer_2024_25/2682_1248.tfw
vorhanden: orthofoto/sommer_2024_25/2683_1246.tif
vorhanden: orthofoto/sommer_2024_25/2683_1246.tfw
vorhanden: orthofoto/sommer_2024_25/2683_1247.tif
vorhanden: orthofoto/sommer_2024_25/2683_1247.tfw
vorhanden: orthofoto/sommer_2024_25/2683_1248.tif
vorhanden: orthofoto/sommer_2024_25/2683_1248.tfw
vorhanden: orthofoto/sommer_2024_25/2684_1246.tif
vorhanden: orthofoto/sommer_2024_25/2684_1246.tfw
vorhanden: orthofoto/sommer_2024_25/2684_1247.tif
vorhanden: orthofoto/sommer_2024_25/2684_1247.tfw
vorhanden: orthofoto/sommer_2024_25/2684_1248.tif
vorhanden: orthofoto/sommer_2024_25/2684_1248.tfw


**Schritt 3: Frühjahr 2021, ohne Laub (Versuch A).** Dieses Bild ist anders aufgeteilt: Die Kacheln sind nummeriert (`2508.tif`) und grösser (etwa 2,2 × 1,5 km, je 3 bis 4 GB). Welche Nummer wo liegt, steht in einem **Kachelverzeichnis** (Shapefile `ortho_fruehjahr21.shp`), das der Kanton mitliefert. Wir laden zuerst das Verzeichnis und wählen daraus die Kacheln, die das Gebiet samt Rand berühren.

Vorher: Kachelverzeichnis des Kantons → nachher: die Kacheln über dem Gebiet in `orthofoto/fruehjahr_2021/`.

In [10]:
ORTHO_2021 = "https://maps.zh.ch/download/orthofoto/fruehjahr/2021/tif/"
for endung in (".shp", ".shx", ".dbf", ".prj"):                         # ein Shapefile besteht aus mehreren kleinen Dateien
    hole(f"orthofoto/fruehjahr_2021/ortho_fruehjahr21{endung}", ORTHO_2021 + "ortho_fruehjahr21" + endung,
         "Kachelverzeichnis Orthofoto Frühjahr 2021", KANTON, LIZENZ_KANTON)
verzeichnis = gpd.read_file(ROH / "orthofoto/fruehjahr_2021/ortho_fruehjahr21.shp").to_crs(CRS)   # 1 Zeile = 1 Kachel
noetig = verzeichnis[verzeichnis.intersects(rand)]                      # Maske: berührt die Kachel das Gebiet samt Rand?
print(f"Vorher: {len(verzeichnis)} Kacheln im Kanton → nachher: {len(noetig)} für das Gebiet:", list(noetig.location))
for datei in noetig.location:                                           # zum Beispiel "2508.tif"
    for endung in (".tif", ".tfw"):
        name = datei.replace(".tif", endung)
        hole(f"orthofoto/fruehjahr_2021/{name}", ORTHO_2021 + name,
             "Orthofoto Frühjahr 2021 (kantonaler Datensatz, laubfrei, 5 cm)", KANTON, LIZENZ_KANTON)

vorhanden: orthofoto/fruehjahr_2021/ortho_fruehjahr21.shp
vorhanden: orthofoto/fruehjahr_2021/ortho_fruehjahr21.shx
vorhanden: orthofoto/fruehjahr_2021/ortho_fruehjahr21.dbf
vorhanden: orthofoto/fruehjahr_2021/ortho_fruehjahr21.prj
Vorher: 675 Kacheln im Kanton → nachher: 6 für das Gebiet: ['2407.tif', '2408.tif', '2507.tif', '2508.tif', '2607.tif', '2608.tif']
vorhanden: orthofoto/fruehjahr_2021/2407.tif
vorhanden: orthofoto/fruehjahr_2021/2407.tfw
vorhanden: orthofoto/fruehjahr_2021/2408.tif
vorhanden: orthofoto/fruehjahr_2021/2408.tfw
vorhanden: orthofoto/fruehjahr_2021/2507.tif
vorhanden: orthofoto/fruehjahr_2021/2507.tfw
vorhanden: orthofoto/fruehjahr_2021/2508.tif
vorhanden: orthofoto/fruehjahr_2021/2508.tfw
vorhanden: orthofoto/fruehjahr_2021/2607.tif
vorhanden: orthofoto/fruehjahr_2021/2607.tfw
vorhanden: orthofoto/fruehjahr_2021/2608.tif
vorhanden: orthofoto/fruehjahr_2021/2608.tfw


**Schritt 4: Sommer 2026, 10 cm (nur für Notebook 01).** Notebook 01 vergleicht drei Aufnahmen derselben Stelle. Für die Versuche A und B wird dieses Bild nicht gebraucht. Es hat dasselbe Kachelsystem wie 2024/25.

Vorher: Liste der Kacheln → nachher: die Dateien in `orthofoto/sommer_2026/`.

In [11]:
ORTHO_2026 = "https://maps.zh.ch/download/orthofoto/sommer/2026/tif/"
for kachel in kacheln:
    for endung in (".tif", ".tfw"):
        hole(f"orthofoto/sommer_2026/{kachel}{endung}", ORTHO_2026 + kachel + endung,
             "Orthofoto Sommer 2026 (kantonaler Datensatz, 10 cm)", KANTON, LIZENZ_KANTON)

vorhanden: orthofoto/sommer_2026/2682_1246.tif
vorhanden: orthofoto/sommer_2026/2682_1246.tfw
vorhanden: orthofoto/sommer_2026/2682_1247.tif
vorhanden: orthofoto/sommer_2026/2682_1247.tfw
vorhanden: orthofoto/sommer_2026/2682_1248.tif
vorhanden: orthofoto/sommer_2026/2682_1248.tfw
vorhanden: orthofoto/sommer_2026/2683_1246.tif
vorhanden: orthofoto/sommer_2026/2683_1246.tfw
vorhanden: orthofoto/sommer_2026/2683_1247.tif
vorhanden: orthofoto/sommer_2026/2683_1247.tfw
vorhanden: orthofoto/sommer_2026/2683_1248.tif
vorhanden: orthofoto/sommer_2026/2683_1248.tfw
vorhanden: orthofoto/sommer_2026/2684_1246.tif
vorhanden: orthofoto/sommer_2026/2684_1246.tfw
vorhanden: orthofoto/sommer_2026/2684_1247.tif
vorhanden: orthofoto/sommer_2026/2684_1247.tfw
vorhanden: orthofoto/sommer_2026/2684_1248.tif
vorhanden: orthofoto/sommer_2026/2684_1248.tfw


## 4. Kontrolle: Liegt alles da?

Wir stellen die Liste aller Dateien zusammen, die dieses Notebook laden soll, und prüfen für jede: Ist sie vorhanden, und steht sie im Manifest?

Vorher: geladene Dateien → nachher: eine Tabelle je Ordner mit Anzahl, Grösse und der Zahl der Einträge im Manifest.

In [12]:
erwartet = [datei for datei, _, _, _, _ in vektoren] + [LIEGENSCHAFTEN]
erwartet += [f"orthofoto/sommer_2024_25/{kachel}{endung}" for kachel in kacheln for endung in (".tif", ".tfw")]
erwartet += [f"orthofoto/sommer_2026/{kachel}{endung}" for kachel in kacheln for endung in (".tif", ".tfw")]
erwartet += [f"orthofoto/fruehjahr_2021/{datei.replace('.tif', endung)}" for datei in noetig.location for endung in (".tif", ".tfw")]

im_manifest = set(pd.read_csv(MANIFEST).datei)                          # alle Dateien, die im Manifest stehen
liste = pd.DataFrame({"datei": erwartet})
liste["ordner"] = liste.datei.str.rsplit("/", n=1).str[0]               # alles vor dem letzten Schrägstrich
liste["vorhanden"] = [(ROH / datei).exists() for datei in erwartet]
liste["im_manifest"] = [f"daten/roh/{datei}" in im_manifest for datei in erwartet]
liste["gb"] = [(ROH / datei).stat().st_size / 1e9 if (ROH / datei).exists() else 0 for datei in erwartet]
print(f"{int(liste.vorhanden.sum())} von {len(liste)} Dateien vorhanden, {int(liste.im_manifest.sum())} davon im Manifest, zusammen {liste.gb.sum():.1f} GB")
liste.groupby("ordner").agg(dateien=("datei", "count"), vorhanden=("vorhanden", "sum"), im_manifest=("im_manifest", "sum"), gb=("gb", "sum")).round(2)

56 von 56 Dateien vorhanden, 56 davon im Manifest, zusammen 33.1 GB


,dateien,vorhanden,im_manifest,gb
ordner,,,,
grenzen,1,1,1,0.00
liegenschaften,1,1,1,0.01
orthofoto/fruehjahr_2021,12,12,12,19.39
orthofoto/sommer_2024_25,18,18,18,8.13
orthofoto/sommer_2026,18,18,18,5.09
parkplaetze,3,3,3,0.02
vermessung,3,3,3,0.47


## Was wir gesehen haben

- Alle Rohdaten, die die Notebooks 00 bis 13 lesen, liegen in `daten/roh/`; jede Datei steht mit Quelle, Lizenz und Fingerabdruck im Manifest.
- Im Manifest stehen ausserdem die Startgewichte des Netzes (Notebook 07 lädt sie beim ersten Lernen) und die PDFs der Literatur (`quellen_liste.py`). Die Zeilen zu Dateien, die nur ein früherer Durchgang verwendet hat, wurden am 27.11.2026 zusammen mit den Dateien entfernt: Notebook 01 prüft jede Zeile des Manifests und würde sie sonst als fehlend melden.
- Die Daten gehören den Anbietern. Bei den Luftbildern des Kantons ist die Quelle zu nennen (`DATEN_LIZENZEN.md`).

## Wie es weitergeht

Notebook 00 schneidet die Vermessungsdaten auf das Gebiet zu und prüft, ob die Luftbilder das Gebiet decken. Notebook 01 prüft die Fingerabdrücke und sieht sich die Rohdaten an.